In [5]:
import os
import glob
import random
import numpy as np
from PIL import Image, ImageOps
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n" + "-"*30)

# --- 1. Data Parsing ---
d = {}
with open('dataset/forms_for_parsing.txt') as f:
    for line in f:
        key = line.split(' ')[0]
        writer = line.split(' ')[1]
        d[key] = writer
print(f"Total writers mapped in dictionary: {len(d.keys())}")

tmp = []
target_list = []

# Update this path if your images are in a different spot!
path_to_files = os.path.join('dataset\data_subset\data_subset', '*') 
all_found_paths = sorted(glob.glob(path_to_files))

print(f"Files found at path '{path_to_files}': {len(all_found_paths)}")

for filename in all_found_paths:
    # Skip directories or hidden system files
    if not os.path.isfile(filename):
        continue
        
    image_name = os.path.basename(filename) 
    file, ext = os.path.splitext(image_name)
    parts = file.split('-')
    
    if len(parts) >= 2:
        form = parts[0] + '-' + parts[1]
        if form in d:
            tmp.append(filename)
            target_list.append(str(d[form]))
        else:
            print(f"  [Warning] Form '{form}' not found in dictionary!")
    else:
        print(f"  [Warning] File '{image_name}' doesn't match the expected naming format.")

img_files = np.asarray(tmp)
img_targets = np.asarray(target_list)

print("-" * 30)
print(f"Final Valid Images: {len(img_files)}")
print(f"Final Valid Labels: {len(img_targets)}")

if len(img_files) == 0:
    print("\n🚨 ERROR: No valid images found! Please double-check your 'dataset/data_subset' folder path.")
else:
    encoder = LabelEncoder()
    encoded_Y = encoder.fit_transform(img_targets)
    num_classes = len(np.unique(encoded_Y))

    train_files, rem_files, train_targets, rem_targets = train_test_split(
            img_files, encoded_Y, train_size=0.66, random_state=52, shuffle=True)

    validation_files, test_files, validation_targets, test_targets = train_test_split(
            rem_files, rem_targets, train_size=0.5, random_state=22, shuffle=True)
    
    print("\n✅ Split Successful!")
    print(f"Train samples: {len(train_files)} | Val samples: {len(validation_files)} | Test samples: {len(test_files)}")

Using device: cuda
------------------------------
Total writers mapped in dictionary: 1539
Files found at path 'dataset\data_subset\data_subset\*': 4899
------------------------------
Final Valid Images: 4899
Final Valid Labels: 4899

✅ Split Successful!
Train samples: 3233 | Val samples: 833 | Test samples: 833


In [6]:
class HandwritingDataset(Dataset):
    def __init__(self, file_paths, targets):
        self.file_paths = file_paths
        self.targets = targets
        # transforms.ToTensor() automatically scales pixels from 0-255 to 0.0-1.0
        # and converts shape to (Channels, Height, Width)
        self.transform = transforms.ToTensor() 

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        label = self.targets[idx]

        im = Image.open(img_path).convert('L') # Grayscale
        cur_width, cur_height = im.size
        
        height_fac = 113 / cur_height
        new_width = int(cur_width * height_fac)
        
        imresize = im.resize((new_width, 113), Image.LANCZOS)
        now_width = imresize.size[0]

        # Take ONE random 113x113 crop per image load
        # (The model will see different crops of the same image across multiple epochs)
        if now_width > 113:
            start = random.randint(0, now_width - 113)
        else:
            start = 0
            # Pad the image with white if it's somehow narrower than 113px
            imresize = ImageOps.pad(imresize, (113, 113), color=255) 

        imcrop = imresize.crop((start, 0, start+113, 113))
        
        img_tensor = self.transform(imcrop)
        label_tensor = torch.tensor(label, dtype=torch.long)
        
        return img_tensor, label_tensor

# Create DataLoaders
batch_size = 8

train_dataset = HandwritingDataset(train_files, train_targets)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = HandwritingDataset(validation_files, validation_targets)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = HandwritingDataset(test_files, test_targets)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [7]:
class HandwritingCNN(nn.Module):
    def __init__(self, num_classes):
        super(HandwritingCNN, self).__init__()
        
        # Conv1: 32 filters, 5x5 kernel, stride 2
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5, stride=2, padding=2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Conv2: 64 filters, 3x3 kernel, stride 1
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Conv3: 128 filters, 3x3 kernel, stride 1
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected Layers
        # After 56x56 is pooled and convolved 3 times, the feature map is 3x3
        self.fc1 = nn.Linear(128 * 3 * 3, 512)
        self.drop1 = nn.Dropout(0.5)
        
        self.fc2 = nn.Linear(512, 256)
        self.drop2 = nn.Dropout(0.5)
        
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Resize from 113x113 to 56x56
        x = F.interpolate(x, size=(56, 56), mode='bilinear', align_corners=False)
        
        # Feature Extraction
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        
        # Flatten
        x = torch.flatten(x, 1)
        
        # Classification
        x = self.drop1(F.relu(self.fc1(x)))
        x = self.drop2(F.relu(self.fc2(x)))
        x = self.fc3(x)
        
        return x

model = HandwritingCNN(num_classes=num_classes).to(device)
print(model)

HandwritingCNN(
  (conv1): Conv2d(1, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1152, out_features=512, bias=True)
  (drop1): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (drop2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=256, out_features=50, bias=True)
)


In [9]:
epochs = 100
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

best_val_loss = float('inf')

for epoch in range(epochs):
    # --- TRAINING ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad() # Clear old gradients
        
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels) # Calculate loss
        loss.backward() # Backpropagation
        optimizer.step() # Update weights
        
        running_loss += loss.item()
        
        # Calculate training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct_train / total_train
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # No gradients needed for validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct_val / total_val
    
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    # Save the best model (equivalent to ModelCheckpoint)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'low_loss_pytorch.pth')
        print("  -> Saved new best model!")

print("Training Complete.")

Epoch 1/100 | Train Loss: 2.3283, Train Acc: 32.20% | Val Loss: 2.2206, Val Acc: 32.89%
  -> Saved new best model!
Epoch 2/100 | Train Loss: 2.2463, Train Acc: 32.60% | Val Loss: 2.1270, Val Acc: 32.53%
  -> Saved new best model!
Epoch 3/100 | Train Loss: 2.2230, Train Acc: 33.87% | Val Loss: 1.8498, Val Acc: 44.42%
  -> Saved new best model!
Epoch 4/100 | Train Loss: 2.1256, Train Acc: 36.10% | Val Loss: 1.9371, Val Acc: 41.30%
Epoch 5/100 | Train Loss: 2.0384, Train Acc: 38.69% | Val Loss: 2.1267, Val Acc: 36.25%
Epoch 6/100 | Train Loss: 2.0274, Train Acc: 37.95% | Val Loss: 1.6867, Val Acc: 47.78%
  -> Saved new best model!
Epoch 7/100 | Train Loss: 1.9268, Train Acc: 40.06% | Val Loss: 1.6048, Val Acc: 48.02%
  -> Saved new best model!
Epoch 8/100 | Train Loss: 1.9012, Train Acc: 41.14% | Val Loss: 1.7046, Val Acc: 45.50%
Epoch 9/100 | Train Loss: 1.8857, Train Acc: 42.34% | Val Loss: 1.6193, Val Acc: 48.02%
Epoch 10/100 | Train Loss: 1.8043, Train Acc: 43.33% | Val Loss: 1.4647, 